<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/5-agent-api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectara Agent API Examples

This notebook demonstrates the basics of how to use Vectara's Agent REST APIs directly to create and interact with AI agents.

You'll learn how to:
1. Create an agent with custom instructions
2. Create agent sessions for conversations
3. Send messages to agents and get responses
4. Use streaming for real-time responses
5. Manage conversation history
6. Work with tools and tool servers

## About Vectara

[Vectara](https://vectara.com/) is an Agent Platform for trusted enterprise AI — a unified Agentic RAG platform with built-in retrieval, orchestration, and governance. See [Notebook 1](1-corpus-creation.ipynb) for the full overview of features and deployment options.

## Getting Started

This notebook assumes you've completed Notebooks 1-2, and potentially 3-4:
- Notebook 1: Created two corpora (ai-research-papers and vectara-docs)
- Notebook 2: Ingested AI research papers and Vectara documentation
- Notebook 3: Deleted documents from a corpus
- Notebook 4: Queried the data with various techniques

Now we'll create agents that can autonomously search and reason across this data.

## Setup

In [1]:
!pip install -q sseclient-py


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import requests
import json
import uuid
from datetime import datetime

# Get credentials from environment variables
api_key = os.environ['VECTARA_API_KEY']

# Get corpus keys from environment (from Notebook 1)
research_corpus_key = 'tutorial-ai-research-papers'
docs_corpus_key = 'tutorial-vectara-docs'

# Base API URL
BASE_URL = "https://api.vectara.io/v2"

# Common headers
headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json"
}

## Step 1: Create a Basic Agent

Create an agent with custom instructions that can search your corpus:

In [3]:
# Define agent configuration - this agent can access both corpora
# Agent structure uses steps with instructions and tool configurations
agent_name = "RAG Research Assistant"
agent_config = {
    "name": agent_name,
    "description": "Agent that can answer questions about RAG, embeddings, and retrieval from both research papers and documentation",
    "model": { "name": "gpt-4o" },
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "first set of instructions",
                    "template": """
You are an expert AI research assistant specializing in Retrieval Augmented Generation and AI Agents. 
You have access to both academic research papers and Vectara's technical documentation. 
Provide clear, accurate answers with citations. 
When answering, combine theoretical insights from research with practical implementation guidance from documentation.
                            """
                }
            ],
            "output_parser": {"type": "default"}
        }
    },
    
    "tool_configurations": {
        "research_paper_search": {
            "type": "corpora_search",
            "query_configuration": {
                "search": {
                    "corpora": [
                        {
                            "corpus_key": research_corpus_key
                        }
                    ]
                },
                "generation": {
                    "generation_preset_name": "vectara-summary-table-md-query-ext-jan-2025-gpt-4o",
                    "model_parameters": {
                        "llm_name": "gpt-4o",
                        "temperature": 0.0
                    }
                }
            }
        },
        "vectara_doc_search": {
            "type": "corpora_search",
            "query_configuration": {
                "search": {
                    "corpora": [
                        {
                            "corpus_key": docs_corpus_key
                        }
                    ]
                },
                "generation": {
                    "generation_preset_name": "vectara-summary-table-md-query-ext-jan-2025-gpt-4o",
                    "model_parameters": {
                        "llm_name": "gpt-4o",
                        "temperature": 0.0
                    }
                }
            }
        }

    }
}

# Inlined here to show the pagination pattern explicitly; notebooks 6+ import
# the same helper as vectara_utils.find_agents_by_name.
def find_agent_by_name(name):
    """Find an agent by name, handling pagination."""
    page_key = None
    while True:
        params = {'limit': 100}
        if page_key:
            params['page_key'] = page_key
        resp = requests.get(f"{BASE_URL}/agents", headers=headers, params=params)
        if resp.status_code != 200:
            break
        data = resp.json()
        for agent in data.get('agents', []):
            if agent.get('name') == name:
                return agent
        page_key = data.get('metadata', {}).get('page_key')
        if not page_key:
            break
    return None

# Check if agent already exists
print(f"Checking if agent '{agent_name}' already exists...")
agent_key = None
existing = find_agent_by_name(agent_name)
if existing:
    agent_key = existing['key']
    print(f"✓ Agent already exists!")
    print(f"  Agent Key: {agent_key}")
    print(f"  Agent Name: {existing['name']}")
    print(f"  Steps: {list(existing.get('steps', {}).keys())}")

# Create the agent only if it doesn't exist
if not agent_key:
    print(f"Creating new agent '{agent_name}'...")
    response = requests.post(f"{BASE_URL}/agents", headers=headers, json=agent_config)
    
    print(f"Status Code: {response.status_code}")
    if response.status_code == 201:
        agent_data = response.json()
        agent_key = agent_data["key"]
        print(f"✓ Agent Created!")
        print(f"  Agent Key: {agent_key}")
        print(f"  Agent Name: {agent_data['name']}")
        print(f"  Steps: {list(agent_data.get('steps', {}).keys())}")
    elif response.status_code == 409:
        # Agent was created between our check and create call; look it up again
        existing = find_agent_by_name(agent_name)
        if existing:
            agent_key = existing['key']
            print(f"✓ Agent already exists!")
            print(f"  Agent Key: {agent_key}")
        else:
            print(f"Error: Agent exists per API but could not be found in listing")
    else:
        print(f"Error: {response.text}")

Checking if agent 'RAG Research Assistant' already exists...


✓ Agent already exists!
  Agent Key: agt_rag_research_assistant_4627
  Agent Name: RAG Research Assistant
  Steps: ['first_step']


## Step 2: Create an Agent Session

Sessions maintain conversation context across multiple turns:

In [4]:
# Create a new session with a unique name to allow reruns
session_name = f"Technical Support Chat {datetime.now().strftime('%Y%m%d-%H%M%S')}"
session_config = {
    "name": session_name,
    "metadata": {
        "user_type": "developer",
        "session_purpose": "api_questions"
    }
}

url = f"{BASE_URL}/agents/{agent_key}/sessions"
response = requests.post(url, headers=headers, json=session_config)

print(f"Status Code: {response.status_code}")
if response.status_code == 201:
    session_data = response.json()
    session_key = session_data["key"]
    print(f"✓ Session Created!")
    print(f"  Session Name: {session_name}")
    print(f"  Session Key: {session_key}")
else:
    print(f"Error: {response.text}")

Status Code: 201
✓ Session Created!
  Session Name: Technical Support Chat 20260528-151501
  Session Key: ase_technical_support_chat_20260528-151501_655e


## Step 3: Send Messages to the Agent

Send a message and get a response:

In [5]:
# Send a message to the agent
# The correct format uses a messages array with message objects
message_data = {
    "messages": [
        {
            "type": "text",
            "content": "What is retrieval augmented generation and how can I implement it with Vectara?"
        }
    ],
    "stream_response": False
}

url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
response = requests.post(url, headers=headers, json=message_data)

if response.status_code == 201:
    event_data = response.json()
    print(f"\nAgent Response:")
    # The response typically contains the assistant's message in the events
    if 'events' in event_data:
        for event in event_data['events']:
            if event.get('type') == 'agent_output':
                print(event.get('content', 'No content'))
    else:
        print(event_data)
else:
    print(f"Error: {response.text}")


Agent Response:
Retrieval Augmented Generation (RAG) is a method that combines pre-trained language models with external retrieval capabilities, allowing these models to access both parametric and non-parametric memory. This approach is particularly suited for knowledge-intensive tasks as it involves retrieving relevant documents or data that can enhance the outputs of language generation models by grounding their responses in retrieved facts [research_paper_1].

To implement RAG using Vectara, you can take advantage of its flexible prompt customization features. Vectara enables the integration of retrieved documents and their metadata into the generation prompts, improving the relevance and accuracy of the generated responses. The platform provides a Custom RAG Prompt Engine that allows developers to craft prompts leveraging diverse prompt variables and functions, adaptable for various applications such as enterprise search, chatbots, or knowledge bases [vectara_doc_1], [vectara_doc_

## Step 4: Multi-Turn Conversation

The agent maintains conversation context automatically:

In [6]:
# First message
message_1 = {
    "messages": [
        {
            "type": "text",
            "content": "What is hybrid search?"
        }
    ],
    "stream_response": False
}

url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
response = requests.post(url, headers=headers, json=message_1)

print("User: What is hybrid search?")

if response.status_code == 201:
    event_data = response.json()
    print(f"\nAgent Response:")
    if 'events' in event_data:
        for event in event_data['events']:
            if event.get('type') == 'agent_output':
                print(event.get('content', 'No content'))
    else:
        print(event_data)
else:
    print(f"Error: {response.text}")
    
print("\n" + "="*80 + "\n")

User: What is hybrid search?

Agent Response:
Hybrid search is an approach that combines the capabilities of both semantic search and keyword-based search. Within the context of Vectara, a hybrid search involves using a semantic index that is both writable and filterable, allowing for efficient retrieval of multilingual data. This type of search can handle structured storage with metadata like user_id, session_id, topic, and indexed_at, blending elements from memory, knowledge, and cache in multi-corpus scenarios. Vectara’s platform supports this setup with built-in chain rerankers, citation capabilities, and role-based access control (RBAC), ensuring users access only the information they are authorized to see [vectara_doc_1], [vectara_doc_2].




In [7]:
# Follow-up message (agent remembers context)
message_2 = {
    "messages": [
        {
            "type": "text",
            "content": "What are its main benefits?"
        }
    ],
    "stream_response": False
}

response = requests.post(url, headers=headers, json=message_2)

print("User: What are its main benefits?")
if response.status_code == 201:
    event_data = response.json()
    print(f"\nAgent Response:")
    if 'events' in event_data:
        for event in event_data['events']:
            if event.get('type') == 'agent_output':
                print(event.get('content', 'No content'))
    else:
        print(event_data)
else:
    print(f"Error: {response.text}")

User: What are its main benefits?

Agent Response:
Hybrid search, particularly as implemented by platforms like Vectara, offers several benefits:

1. **Comprehensive Search Coverage**: By combining semantic search with keyword-based search, hybrid search ensures a more thorough and nuanced approach to finding information, capturing both the context and specific keywords in a query [vectara_doc_2].

2. **Enhanced Precision and Relevance**: The use of semantic understanding allows hybrid search to consider the meaning and context behind words, improving the relevance of search results compared to traditional keyword-only searches [vectara_doc_1].

3. **Structured Data Retrieval**: Hybrid search supports the use of metadata and structured filters (such as user_id, topic, and session_id) that enhance the ability to retrieve precise and contextually relevant information, enabling more targeted and efficient searches [vectara_doc_1].

4. **Multilingual Capabilities**: The inclusion of semant

In [8]:
# Another follow-up
message_3 = {
    "messages": [
        {
            "type": "text",
            "content": "Can you give me an example?"
        }
    ],
    "stream_response": False
}

response = requests.post(url, headers=headers, json=message_3)

print("User: Can you give me an example?")
if response.status_code == 201:
    event_data = response.json()
    print(f"\nAgent Response:")
    # The response typically contains the assistant's message in the events
    if 'events' in event_data:
        for event in event_data['events']:
            if event.get('type') == 'agent_output':
                print(event.get('content', 'No content'))
    else:
        print(event_data)
else:
    print(f"Error: {response.text}")

User: Can you give me an example?

Agent Response:
Certainly! Let's consider an example of how hybrid search could be used in a real-world scenario:

**Scenario: Customer Support System**

Imagine a large company that provides customer support across multiple countries. Their support system needs to handle a variety of customer queries, which can be highly specific or very general. The company has a comprehensive database of support documents, past ticket resolutions, product manuals, and FAQs.

**Hybrid Search Benefits:**

1. **Semantic Understanding**: When a customer inputs a query like, "I can't connect my printer to Wi-Fi," hybrid search uses semantic understanding to relate this to similar issues, broader connectivity problems, and specific printer models mentioned across documents.

2. **Keyword Matching**: At the same time, the system utilizes keyword-based matching to surface documents specifically mentioning "printer" and "Wi-Fi," ensuring that the relevant details are priori

## Step 5: Streaming Responses

Vectara's Agent API supports Server-Sent Events (SSE) for streaming responses in real time. Instead of waiting for the complete response, you can process text chunks, tool calls, and thinking events as they arrive. Set `stream_response: True` in your message payload to enable streaming.

In [9]:
import sseclient

# Send a message with streaming enabled
message_data = {
    "messages": [
        {
            "type": "text",
            "content": "Briefly explain how embeddings work in retrieval systems."
        }
    ],
    "stream_response": True
}

url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
response = requests.post(url, headers=headers, json=message_data, stream=True)

print("Streaming response:\n")

# The events endpoint returns 201 Created for both JSON and SSE (streaming) responses,
# so we check response.ok rather than a specific status code.
if response.ok:
    client = sseclient.SSEClient(response)
    for sse_event in client.events():
        try:
            event = json.loads(sse_event.data)
        except json.JSONDecodeError:
            continue
        event_type = event.get("type", "")

        if event_type == "streaming_agent_output":
            # Print each text chunk as it arrives
            print(event.get("content", ""), end="", flush=True)
        elif event_type == "streaming_agent_output_end":
            print("\n\n--- Stream complete ---")
        elif event_type == "tool_input":
            tool = event.get("tool_configuration_name", "unknown")
            print(f"[Calling tool: {tool}]")
        elif event_type == "tool_output":
            tool = event.get("tool_configuration_name", "unknown")
            print(f"[Tool response received: {tool}]")
        elif event_type == "thinking":
            print(f"[Thinking: {event.get('content', '')[:80]}...]")
else:
    print(f"Error {response.status_code}: {response.text}")

Streaming response:

[Calling tool: research_paper_search]


[Tool response received: research_paper_search]


Emb

eddings

 work

...

aper

_

1

].



--- Stream complete ---
